In [ ]:
# # Problem 2 — Structured Summarizer with Evaluation

# ## Architecture
# ```
# Sample Documents
#     │
#     ├── Prompt V1 (basic)  ── LLM ──► Summary V1
#     └── Prompt V2 (rich)   ── LLM ──► Summary V2
#                                           │
#     ┌─────────────────────────────────────┘
#     ▼
# LLM-as-Judge → Clarity / Completeness Scores → Comparison Report
# ```

# ## Stack
# - **LLM**: OpenAI (gpt-4o)
# - **Prompt management**: Langfuse Prompt Registry
# - **Observability**: Langfuse v4 (auto-tracing via langfuse.openai wrapper)
# - **Evaluation**: LLM-as-Judge

In [ ]:
## Step 1 — Install Required Libraries
!pip install -q openai langfuse pypdf pandas langchain-openai
## Step 2 — Import Dependencies
import os
import json
import pandas as pd
from datetime import datetime
from pypdf import PdfReader

# Langfuse v4 — use the wrapped OpenAI client for auto-tracing
from langfuse import Langfuse
from langfuse.openai import OpenAI   # drop-in replacement — auto-traces every call

print("All dependencies imported ✅")

In [4]:



## Step 3 — Configure Environment Variables
# ── OpenAI ────────────────────────────────────────────────────
os.environ["OPENAI_API_KEY"]      = "sk-YOUR_OPENAI_KEY"

# ── Langfuse ──────────────────────────────────────────────────
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-YOUR_PUBLIC_KEY"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-YOUR_SECRET_KEY"
os.environ["LANGFUSE_HOST"]       = "https://cloud.langfuse.com"

MODEL_NAME   = "gpt-4o"
PROMPT_NAME  = "structured-summarizer"
DATASET_NAME = "summarizer-eval-dataset"

print("Environment configured ✅")
## Step 4 — Initialize Clients

> **Key point for Langfuse v4**: Use `lf_client` (the `Langfuse()` instance) for all SDK calls.
> Use `langfuse.openai.OpenAI` (wrapped client) for all LLM calls — this auto-traces to Langfuse.
# Langfuse client — for prompt registry, dataset management, scoring
# Named lf_client to avoid conflict with the langfuse MODULE
lf_client = Langfuse(
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    host=os.environ["LANGFUSE_HOST"]
)

# OpenAI client — wrapped by Langfuse for automatic tracing
# Every .chat.completions.create() call is auto-traced — no manual spans needed
ai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print(f"Langfuse client ready: {type(lf_client).__name__} ✅")
print(f"OpenAI client ready:   {type(ai_client).__name__} ✅")
## Step 5 — Register Two Prompt Versions in Langfuse

| | V1 | V2 |
|---|---|---|
| Structure | Free-form prose | Explicit JSON schema |
| Sections | Unspecified | Mandatory 5 sections |
| Tone | Generic | Professional, formal |
# ── V1: Minimal prompt ────────────────────────────────────────
PROMPT_V1_TEXT = """Summarize the following document.

Document:
{{document_text}}
"""

# ── V2: Structured JSON schema prompt ─────────────────────────
PROMPT_V2_TEXT = """You are an expert technical writer producing an executive-level document summary.

Analyse the document below and return a structured JSON summary with these exact keys:

{
  "title":        "<inferred document title>",
  "overview":     "<2-3 sentence high-level summary>",
  "key_points":   ["<point 1>", "<point 2>", "<point 3>", "<point 4>", "<point 5>"],
  "conclusion":   "<1-2 sentence synthesis of the main takeaway>",
  "action_items": ["<concrete next step 1>", "<concrete next step 2>"]
}

Rules:
- Return ONLY valid JSON. No markdown fences, no extra text.
- Each key_point must be a standalone, self-explanatory sentence.
- Avoid generic filler. Every sentence should carry factual content.
- Use formal professional tone.

Document:
{{document_text}}
"""

# Register V1 in Langfuse Prompt Registry
lf_client.create_prompt(
    name=PROMPT_NAME,
    prompt=PROMPT_V1_TEXT,
    labels=["v1", "baseline"],
    config={"model": MODEL_NAME, "temperature": 0}
)

# Register V2 in Langfuse Prompt Registry
lf_client.create_prompt(
    name=PROMPT_NAME,
    prompt=PROMPT_V2_TEXT,
    labels=["v2", "structured"],
    config={"model": MODEL_NAME, "temperature": 0}
)

print(f"Prompts registered under name='{PROMPT_NAME}' in Langfuse ✅")
## Step 6 — Sample Documents for Evaluation
SAMPLE_DOCS = [
    {
        "id": "doc-001",
        "text": (
            "Artificial intelligence (AI) is rapidly transforming the healthcare industry. "
            "Machine learning algorithms can now analyse medical images with accuracy that rivals "
            "expert radiologists. Natural language processing tools extract clinical insights from "
            "unstructured doctor notes, reducing administrative burden by up to 40%. "
            "Predictive models identify high-risk patients before hospitalisation, enabling "
            "early interventions that reduce readmission rates. However, challenges remain: "
            "data privacy regulations (HIPAA, GDPR), algorithmic bias in underrepresented "
            "populations, and the critical need for clinical validation before deployment. "
            "Healthcare organisations should prioritise AI governance frameworks, invest in "
            "diverse training data, and establish cross-functional AI ethics committees."
        )
    },
    {
        "id": "doc-002",
        "text": (
            "The global supply chain disruption of 2020-2023 exposed significant vulnerabilities "
            "in just-in-time manufacturing. Companies relying on single-source suppliers experienced "
            "production halts lasting months. In response, leading manufacturers are adopting "
            "supply chain digitalisation strategies: real-time inventory tracking via IoT sensors, "
            "demand forecasting using ML models trained on 5+ years of historical data, and "
            "supplier risk scoring using ESG and financial health indicators. "
            "Firms that implemented dual-sourcing strategies reduced stockout risk by 35%. "
            "The recommended action is to conduct a supply chain resilience audit and identify "
            "the top 10 single points of failure within the next quarter."
        )
    },
    {
        "id": "doc-003",
        "text": (
            "Schneider Electric's EcoStruxure platform connects over 70 million active installations "
            "across 115 countries. The platform integrates edge devices, software, and analytics "
            "to optimise energy consumption in buildings, data centres, and industrial facilities. "
            "In 2023, EcoStruxure-enabled buildings reported average energy savings of 27%. "
            "The industrial segment leverages predictive maintenance AI that reduces unplanned "
            "downtime by up to 70%. Key differentiators include open standards, cybersecurity "
            "compliance (IEC 62443), and seamless integration with third-party SCADA and MES systems. "
            "Expansion into AI-powered grid management is planned for 2025-2026."
        )
    },
]

print(f"Sample documents ready: {len(SAMPLE_DOCS)} docs ✅")
## Step 7 — Load PDF (Optional)

If you have a PDF, extract text here and add it to `SAMPLE_DOCS`.
def extract_pdf_text(pdf_path: str) -> str:
    """Extract all text from a PDF using pypdf."""
    reader = PdfReader(pdf_path)
    pages_text = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            pages_text.append(f"--- Page {i+1} ---\n{text}")
    full_text = "\n".join(pages_text)
    print(f"Extracted {len(reader.pages)} pages, {len(full_text)} characters")
    return full_text

# ── Uncomment to load your PDF ────────────────────────────────
# PDF_PATH = "path/to/your/document.pdf"
# pdf_text = extract_pdf_text(PDF_PATH)
# SAMPLE_DOCS.append({"id": "doc-pdf", "text": pdf_text[:8000]})
# print("PDF added ✅")

print("PDF loader ready. Uncomment to use.")
## Step 8 — Define Prompt V1 and V2 Functions

Each function:
1. Fetches the registered prompt from Langfuse
2. Compiles it with the document text
3. Calls OpenAI (auto-traced by Langfuse v4)
4. Returns the output
def prompt_v1(document_text: str):
    """
    Summarize using Prompt V1 — basic, unstructured.
    Returns (summary_text, prompt_label)
    """
    # Fetch registered prompt from Langfuse
    lf_prompt = lf_client.get_prompt(PROMPT_NAME, label="v1")

    # Compile: replace {{document_text}} placeholder
    compiled = lf_prompt.prompt.replace("{{document_text}}", document_text)

    # Call LLM — auto-traced by Langfuse v4 wrapped client
    response = ai_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": compiled}],
        temperature=0,
        name="summarizer-v1"   # trace name in Langfuse dashboard
    )
    return response.choices[0].message.content, "v1"


def prompt_v2(document_text: str):
    """
    Summarize using Prompt V2 — structured JSON schema.
    Returns (summary_dict_or_text, prompt_label)
    """
    lf_prompt = lf_client.get_prompt(PROMPT_NAME, label="v2")
    compiled  = lf_prompt.prompt.replace("{{document_text}}", document_text)

    response = ai_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": compiled}],
        temperature=0,
        name="summarizer-v2"
    )
    output = response.choices[0].message.content

    # Try to parse JSON output from V2
    try:
        return json.loads(output), "v2"
    except json.JSONDecodeError:
        return output, "v2"


print("Prompt V1 and V2 functions defined ✅")
## Step 9 — Run Experiment (V1 vs V2 on all documents)
def run_experiment():
    """
    Run both prompt versions on every sample document.
    Returns a DataFrame with outputs for comparison.
    """
    results = []
    print(f"Running experiment on {len(SAMPLE_DOCS)} documents...")

    for doc in SAMPLE_DOCS:
        print(f"  Processing {doc['id']}...")

        # Run V1
        v1_output, _ = prompt_v1(doc["text"])

        # Run V2
        v2_output, _ = prompt_v2(doc["text"])

        results.append({
            "doc_id":     doc["id"],
            "document":   doc["text"][:120] + "...",
            "v1_output":  v1_output,
            "v2_output":  v2_output,
        })

    # Flush all telemetry to Langfuse
    lf_client.flush()
    print("\nAll traces flushed to Langfuse ✅")
    return pd.DataFrame(results)


results_df = run_experiment()
print(f"\nExperiment complete: {len(results_df)} documents processed ✅")
# Preview V1 output
print("=" * 70)
print("V1 OUTPUT — doc-001 (unstructured):")
print(results_df.loc[0, "v1_output"])

print("\n" + "=" * 70)
print("V2 OUTPUT — doc-001 (structured JSON):")
v2 = results_df.loc[0, "v2_output"]
if isinstance(v2, dict):
    print(json.dumps(v2, indent=2))
else:
    print(v2)
## Step 10 — LLM-as-Judge Evaluation

| Metric | Definition |
|---|---|
| **Clarity** | Is the summary easy to understand? Free of jargon and ambiguity? |
| **Completeness** | Does it capture all major points from the source? |
LLM_JUDGE_SYSTEM_PROMPT = """\
You are an expert evaluator assessing the quality of document summaries.

You will be given:
1. Original document text
2. A generated summary

Score the summary on TWO dimensions (1-5 scale each):

Clarity (1-5):
  1 = incomprehensible or very confusing
  3 = understandable but with some ambiguity
  5 = exceptionally clear, concise, professional

Completeness (1-5):
  1 = misses most key points from the source
  3 = covers main topic but misses several important details
  5 = captures all key facts, entities, and conclusions from the source

Return ONLY a JSON object in this exact format (no extra text):
{
  "clarity": <int 1-5>,
  "completeness": <int 1-5>,
  "reasoning": "<one sentence justifying both scores>"
}
"""

def evaluate_summary(document_text: str, summary) -> dict:
    """Use LLM-as-Judge to score a summary on clarity and completeness."""
    summary_str = json.dumps(summary, indent=2) if isinstance(summary, dict) else summary

    user_message = f"""ORIGINAL DOCUMENT:\n{document_text[:3000]}\n\nGENERATED SUMMARY:\n{summary_str}"""

    response = ai_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": LLM_JUDGE_SYSTEM_PROMPT},
            {"role": "user",   "content": user_message}
        ],
        temperature=0,
        name="llm-judge"
    )
    output = response.choices[0].message.content.strip()
    try:
        return json.loads(output)
    except json.JSONDecodeError:
        return {"clarity": None, "completeness": None, "reasoning": output}


print("LLM-as-Judge evaluator defined ✅")
# Run evaluations on all documents
eval_records = []

for _, row in results_df.iterrows():
    # Get full document text
    doc_full = next((d["text"] for d in SAMPLE_DOCS if d["id"] == row["doc_id"]), "")

    print(f"Evaluating {row['doc_id']}...")

    v1_scores = evaluate_summary(doc_full, row["v1_output"])
    v2_scores = evaluate_summary(doc_full, row["v2_output"])

    eval_records.append({
        "doc_id":          row["doc_id"],
        "v1_clarity":      v1_scores.get("clarity"),
        "v1_completeness": v1_scores.get("completeness"),
        "v1_reasoning":    v1_scores.get("reasoning"),
        "v2_clarity":      v2_scores.get("clarity"),
        "v2_completeness": v2_scores.get("completeness"),
        "v2_reasoning":    v2_scores.get("reasoning"),
    })

lf_client.flush()

eval_df = pd.DataFrame(eval_records)
print("\nEvaluation complete ✅")
print(eval_df[["doc_id","v1_clarity","v1_completeness","v2_clarity","v2_completeness"]].to_string(index=False))
## Step 11 — Compare V1 vs V2 Results
# Aggregate score comparison
comparison = pd.DataFrame({
    "Metric": ["Avg Clarity", "Avg Completeness", "Avg Overall"],
    "V1 (Baseline)": [
        round(eval_df["v1_clarity"].mean(), 2),
        round(eval_df["v1_completeness"].mean(), 2),
        round(((eval_df["v1_clarity"] + eval_df["v1_completeness"]) / 2).mean(), 2),
    ],
    "V2 (Structured)": [
        round(eval_df["v2_clarity"].mean(), 2),
        round(eval_df["v2_completeness"].mean(), 2),
        round(((eval_df["v2_clarity"] + eval_df["v2_completeness"]) / 2).mean(), 2),
    ],
})

comparison["Δ (V2 - V1)"] = (
    comparison["V2 (Structured)"] - comparison["V1 (Baseline)"]
).round(2)

print("\n" + "=" * 55)
print("         V1 vs V2 COMPARISON SUMMARY")
print("=" * 55)
print(comparison.to_string(index=False))
print("=" * 55)
# Per-document breakdown
print("Per-document scores:")
print(eval_df[["doc_id","v1_clarity","v1_completeness","v2_clarity","v2_completeness"]].to_string(index=False))

print("\nReasoning from LLM Judge:")
for _, row in eval_df.iterrows():
    print(f"\n[{row['doc_id']}]")
    print(f"  V1: {row['v1_reasoning']}")
    print(f"  V2: {row['v2_reasoning']}")
## V1 vs V2 — Improvement Reasoning

| Dimension | V1 | V2 |
|---|---|---|
| Output format | Unstructured prose | Validated JSON |
| Section coverage | No guarantee | Mandatory 5 sections |
| Action items | Often absent | Always present |
| Tone instruction | None | Professional, formal |
| Machine-parseable | ❌ | ✅ |

**Why V2 wins:**
- Explicit JSON schema forces the model to cover all sections
- 5 mandatory key_points prevent the model from stopping at the first paragraph
- Strict output format rules prevent extra words that break downstream parsing
- Small prompt cost increase (~15 tokens) yields significantly higher quality

## Step 12 — View Traces in Langfuse

Go to [cloud.langfuse.com](https://cloud.langfuse.com) and check:
- **Traces** → see every LLM call with input/output, tokens, latency
- **Prompt Registry** → `structured-summarizer` → compare V1 vs V2 versions
- Filter by `name = summarizer-v1` or `summarizer-v2` to compare side by side